In [ ]:
# Imports
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score
from fairlearn.reductions import ExponentiatedGradient, TruePositiveRateParity
from fairlearn.metrics import (MetricFrame, false_negative_rate, false_positive_rate,
                               selection_rate, count
                               )
from ucimlrepo import fetch_ucirepo


In [14]:
# Data preprocessing 2.0
def split_data(X, y, strata, seed, drop_sensitive=True):
    X_model = X.drop(columns="sex") if drop_sensitive else X.copy()
    idx_train, idx_test = train_test_split(
        X.index, test_size=0.2, stratify=strata, random_state=seed
    )
    y_bin = (y["G3"] < 10).astype(int)
    return (X_model.loc[idx_train], X_model.loc[idx_test],
            y_bin.loc[idx_train], y_bin.loc[idx_test])

def build_preprocessor(columns, var_data):
    meta = var_data[(var_data["role"] == "Feature") & (var_data["name"].isin(columns))]
    cat = meta.loc[meta["type"] == "Categorical", "name"].tolist()
    binary = meta.loc[meta["type"] == "Binary", "name"].tolist()
    num = meta.loc[meta["type"] == "Integer", "name"].tolist()

    missing = set(columns) - set(cat + binary + num)
    assert not missing, f"Columns not covered by any transformer: {missing}"

    return ColumnTransformer([
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
        ("binary", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="if_binary"), binary),
        ("numeric", StandardScaler(), num),
    ]).set_output(transform="pandas")

def build_pipeline(columns, var_data, seed):
    return Pipeline([
        ("prep", build_preprocessor(columns, var_data)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=seed)),
    ])


def make_folds(X, stratify_method, n_splits, seed):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(cv.split(X, stratify_method))

In [15]:
# Hyperparameter Tuning

def tune_baseline(X_train, y_train, pipeline, folds):
    search = GridSearchCV(
        pipeline,
        {"clf__C": [0.01, 0.1, 1, 10]},
        scoring="average_precision",
        cv=folds,
    )
    search.fit(X_train, y_train)
    return search.best_estimator_



In [16]:
METRICS = {
    "accuracy": accuracy_score,
    "balanced_accuracy": balanced_accuracy_score,
    "precision": lambda y, p: precision_score(y, p, zero_division=0),
    "recall": lambda y, p: recall_score(y, p, zero_division=0),
    "fpr": false_positive_rate,
    "fnr": false_negative_rate,
    "sel": selection_rate,
}

def evaluate(y_true, y_pred, sensitive):
    mf = MetricFrame(metrics=METRICS, y_true=y_true, y_pred=y_pred,
                     sensitive_features=sensitive)
    row = mf.overall.to_dict()                       # overall metrics
    for g, vals in mf.by_group.iterrows():           # per-group: recall_F, recall_M, ...
        row.update({f"{m}_{g}": v for m, v in vals.items()})
    row.update({f"{m}_gap": v for m, v in mf.difference(method="between_groups").items()})
    return row

In [17]:
# Model evaluation

FAIRNESS_METRICS = {
    "n": count,
    "fnr": false_negative_rate,
    "fpr": false_positive_rate,
    "sel": selection_rate,
}

OUTPUT_METRICS = {
    "accuracy": accuracy_score,
    "balanced_accuracy": balanced_accuracy_score,
    "precision": lambda y, p: precision_score(y, p, zero_division=0),
    "recall": lambda y, p: recall_score(y, p, zero_division=0),
}

def assess_output_quality(y_true, y_pred):
    out = {name: fn(y_true, y_pred) for name, fn in OUTPUT_METRICS.items()}
    return out


def assess_fairness(y_true, y_pred, sensitive):
    """Per-group metrics. `sensitive` is a Series (sex) or a DataFrame (intersectional)."""
    mf = MetricFrame(metrics=FAIRNESS_METRICS, y_true=y_true, y_pred=y_pred,
                     sensitive_features=sensitive)
    out = {"overall": mf.overall, "by_group": mf.by_group}
    
    return out

In [ ]:
# Full pipeline
## Run for 10 different seeds

seeds = [42 + i for i in range(10)]

#fetch dataset
student_performance = fetch_ucirepo(id=320)

# data (as pandas dataframes)
X = student_performance.data.features
y = student_performance.data.targets
variable_data = student_performance.variables

# Stratify by target + sex
strata = (y["G3"] < 10).astype(int).astype(str) + "_" + X["sex"]

results = []
for run in range(10):
    print("Current run: ", run+1)
    seed = seeds[run]

    # Split data into train and test data
    X_train, X_test, y_train, y_test = split_data(X, y, strata, seed)
    sensitive_features_train = X.loc[X_train.index, "sex"]
    sensitive_features_test = X.loc[X_test.index, "sex"]

    # Build preprocessing Pipeline
    pipeline = build_pipeline(X_train.columns, variable_data, seed)

    # Create validation folds and baseline model
    folds = make_folds(X_train, strata.loc[X_train.index], n_splits=5, seed=seed)

    # Tune baseline using 5-fold validation
    tuned_baseline = tune_baseline(X_train, y_train, pipeline, folds=folds)

    # Predict on test set using tuned baseline
    y_pred_baseline = tuned_baseline.predict(X_test)

    # Apply fairness intervention and predict on test set
    fairness_intervention = ExponentiatedGradient(
        estimator=clone(tuned_baseline),
        constraints=TruePositiveRateParity(),
        sample_weight_name="clf__sample_weight",
    )

    fairness_intervention.fit(X_train, y_train, sensitive_features=sensitive_features_train)
    y_pred_intervention = fairness_intervention.predict(X_test, random_state=seed)

    # Get output and fairness metrics for the baseline and intervention
    results.append({"seed": seed, "model": "baseline",     **evaluate(y_test, y_pred_baseline, sensitive_features_test)})
    results.append({"seed": seed, "model": "intervention", **evaluate(y_test, y_pred_intervention, sensitive_features_test)})

    print("-" * 50)

# Collect and print results
results_df = pd.DataFrame(results)
summary = results_df.drop(columns="seed").groupby("model").agg(["mean", "std"])
print(summary.T.round(3).to_string())


Current run:  1
--------------------------------------------------
Current run:  2
--------------------------------------------------
Current run:  3
--------------------------------------------------
Current run:  4
--------------------------------------------------
Current run:  5
--------------------------------------------------
Current run:  6
--------------------------------------------------
Current run:  7
--------------------------------------------------
Current run:  8
--------------------------------------------------
Current run:  9
--------------------------------------------------
Current run:  10
--------------------------------------------------
model                       baseline  intervention
accuracy              mean     0.788         0.790
                      std      0.031         0.032
balanced_accuracy     mean     0.756         0.755
                      std      0.033         0.033
precision             mean     0.398         0.401
                      s